# Daily Challenge – Week 3, Day 2: Data Handling and Analysis in Python
Dataset: **Data Science Job Salary** (`datascience_salaries.csv`, in this folder).
Tasks: 1) Min-Max normalize `salary` · 2) reduce dimensions with PCA · 3) average & median salary per `experience_level` · 4) summary of insights.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA

# The file's first column is an old row number, so we use it as the index
df = pd.read_csv("datascience_salaries.csv", index_col=0)
print(df.shape)
df.head()

In [ ]:
# Quick check of the data
print("Missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())
print(df["salary_currency"].value_counts())
df["salary"].describe()

## 1. Min-Max normalization of `salary`
Formula: (salary − min) / (max − min) → every salary ends up between 0 and 1.

In [ ]:
scaler = MinMaxScaler()
df["salary_normalized"] = scaler.fit_transform(df[["salary"]])

df[["salary", "salary_normalized"]].describe().round(3)

## 2. Dimensionality reduction with PCA
PCA only works on numbers, so first the text columns are one-hot encoded.
`location` has 320 different cities, so only the 10 most common are kept and the rest become `"Other"` (otherwise we would create 320 columns).

In [ ]:
features = df[["job_title", "job_type", "experience_level", "salary_currency", "location"]].copy()

top_locations = features["location"].value_counts().head(10).index
features.loc[~features["location"].isin(top_locations), "location"] = "Other"

encoded = pd.get_dummies(features, dtype=int)
encoded["salary_normalized"] = df["salary_normalized"]

print("Number of features before PCA:", encoded.shape[1])

In [ ]:
pca = PCA(n_components=2)
components = pca.fit_transform(encoded)

df["PC1"] = components[:, 0]
df["PC2"] = components[:, 1]

print("Features after PCA: 2")
print("Variance explained by each component:", pca.explained_variance_ratio_.round(3))
print("Total variance kept:", round(pca.explained_variance_ratio_.sum(), 3))

In [ ]:
plt.figure(figsize=(8, 6))
for level in ["Entry", "Mid", "Senior", "Executive"]:
    subset = df[df["experience_level"] == level]
    plt.scatter(subset["PC1"], subset["PC2"], label=level, alpha=0.6)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Jobs in 2 dimensions after PCA, colored by experience level")
plt.legend()
plt.show()

## 3. Average and median salary by experience level

In [ ]:
order = ["Entry", "Mid", "Senior", "Executive"]
salary_by_level = (df.groupby("experience_level")["salary"]
                     .agg(["mean", "median", "count"])
                     .reindex(order)
                     .round(0))
salary_by_level

In [ ]:
salary_by_level[["mean", "median"]].plot(kind="bar", figsize=(8, 5))
plt.title("Average vs. median salary by experience level")
plt.ylabel("Salary")
plt.xticks(rotation=0)
plt.show()

## 4. Summary of insights
- **Salary rises with experience up to Senior:** the median goes from about 30,000 (Entry) to 51,000 (Mid) to 68,000 (Senior).
- **Executive is the surprise:** its average (~76,000) is close to Senior, but its **median is only ~46,000**. With just **13 executives**, a few very high salaries pull the average up — the median is the more honest number here, and the group is too small to draw firm conclusions.
- **Average vs. median:** for every level the average is higher than the median, so salaries are **skewed to the right** (a few high earners). This is exactly why both measures are worth reporting.
- **The data is unbalanced:** 727 of the 1,171 jobs are Senior, so overall averages mostly describe senior roles.
- **PCA:** 26 encoded features were reduced to 2 components, which keep about 38% of the variance. That is a modest amount, which is expected for data that is mostly categorical — PCA is most useful here for a quick visual overview, not as a full summary of the data.
- **Data quality notes:** 345 rows are exact duplicates of others (could be repeated postings), and 14 salaries are in EUR or GBP, not USD. For a deeper analysis I would remove duplicates and convert all salaries to one currency before comparing.